In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

raw_responses = spark.table(
    get_table("bronze", "raw_api_responses")
)

catalog_raw = (
    raw_responses
    .filter(F.col("event_type") == "catalog_set")
    .select(
        "execution_id",
        "observed_at",
        "product_name",
        "product_slug",
        "category",
        F.col("response.data").alias("catalog_data")
    )
)

display(catalog_raw)

In [0]:
item_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("slug", T.StringType()),
    T.StructField("gameRef", T.StringType()),
    T.StructField("tags", T.ArrayType(T.StringType())),
    T.StructField("setRoot", T.BooleanType()),
    T.StructField("setParts", T.ArrayType(T.StringType())),
    T.StructField("quantityInSet", T.IntegerType()),
    T.StructField("ducats", T.IntegerType()),
    T.StructField("reqMasteryRank", T.IntegerType()),
    T.StructField("tradingTax", T.IntegerType()),
    T.StructField("tradable", T.BooleanType()),
    T.StructField(
        "i18n",
        T.StructType([
            T.StructField(
                "en",
                T.StructType([
                    T.StructField("name", T.StringType()),
                    T.StructField("description", T.StringType()),
                    T.StructField("wikiLink", T.StringType()),
                    T.StructField("icon", T.StringType()),
                    T.StructField("thumb", T.StringType()),
                    T.StructField("subIcon", T.StringType())
                ])
            )
        ])
    )
])

catalog_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("items", T.ArrayType(item_schema))
])

In [0]:

catalog_parsed = (
    catalog_raw
    .withColumn("catalog", F.from_json("catalog_data", catalog_schema))
)

invalid_catalogs = (
    catalog_parsed
    .filter(
        F.col("catalog").isNull()
        | F.col("catalog.items").isNull()
        | (F.size("catalog.items") == 0)
    )
    .count()
)

if invalid_catalogs > 0:
    raise ValueError(
        f"{invalid_catalogs} catalog responses are invalid or empty."
    )

catalog_exploded = (
    catalog_parsed
    .withColumn("item", F.explode("catalog.items"))
)


In [0]:
stg_wfm_catalog = catalog_exploded.select(
    "execution_id",
    F.to_timestamp("observed_at").alias("observed_at"),
    "product_name",
    "product_slug",
    "category",

    F.col("item.id").alias("item_id"),
    F.col("item.slug").alias("item_slug"),
    F.col("item.gameRef").alias("game_ref"),
    F.col("item.tags").alias("tags"),
    F.col("item.setRoot").alias("is_set_root"),
    F.col("item.setParts").alias("set_part_ids"),
    F.col("item.quantityInSet").alias("quantity_in_set"),
    F.col("item.ducats").alias("ducats"),
    F.col("item.reqMasteryRank").alias("required_mastery_rank"),
    F.col("item.tradingTax").alias("trading_tax"),
    F.col("item.tradable").alias("is_tradable"),

    F.col("item.i18n.en.name").alias("item_name"),
    F.col("item.i18n.en.description").alias("description"),
    F.col("item.i18n.en.wikiLink").alias("wiki_url"),
    F.col("item.i18n.en.icon").alias("icon_path"),
    F.col("item.i18n.en.thumb").alias("thumbnail_path"),
    F.col("item.i18n.en.subIcon").alias("sub_icon_path")
)

display(stg_wfm_catalog)

In [0]:

# Data quality checks

if stg_wfm_catalog.isEmpty():
    raise ValueError("Catalog staging contains no records.")

# Required fields
required_columns = [
    "execution_id",
    "item_id",
    "item_slug",
    "item_name"
]

invalid_records = (
    stg_wfm_catalog
    .filter(
        F.expr(
            " OR ".join(f"`{column}` IS NULL" for column in required_columns)
        )
    )
    .count()
)

if invalid_records > 0:
    raise ValueError(
        f"Catalog contains {invalid_records} records with null required fields."
    )

# Composite key uniqueness
duplicate_keys = (
    stg_wfm_catalog
    .groupBy("execution_id", "item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_keys > 0:
    raise ValueError(
        f"Catalog contains {duplicate_keys} duplicate execution_id + item_id keys."
    )

# Product-level validation
product_validation = (
    stg_wfm_catalog
    .groupBy("execution_id", "product_slug")
    .agg(
        F.count("*").alias("total_items"),
        F.countDistinct("item_id").alias("distinct_items"),
        F.sum(
            F.when(F.col("is_set_root") == True, 1).otherwise(0)
        ).alias("set_roots")
    )
)

display(product_validation)

print("Catalog staging validations passed.")


In [0]:

# Persist staging table

from delta.tables import DeltaTable

target_table = get_table("staging", "stg_wfm_catalog")

merge_condition = """
    target.execution_id = source.execution_id
    AND target.item_id = source.item_id
"""

if spark.catalog.tableExists(target_table):
    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(
            stg_wfm_catalog.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        stg_wfm_catalog.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(target_table)
    )

print(f"Table persisted: {target_table}")


In [0]:

# Validate persisted staging table

stg_catalog_saved = spark.table(target_table)

saved_duplicates = (
    stg_catalog_saved
    .groupBy("execution_id", "item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if saved_duplicates > 0:
    raise ValueError(
        f"Persisted catalog contains {saved_duplicates} duplicate keys."
    )

print(f"Total records: {stg_catalog_saved.count():,}")
print(
    f"Total executions: "
    f"{stg_catalog_saved.select('execution_id').distinct().count():,}"
)
print(
    f"Distinct items: "
    f"{stg_catalog_saved.select('item_id').distinct().count():,}"
)

display(
    stg_catalog_saved
    .groupBy("execution_id", "product_name", "product_slug")
    .agg(
        F.count("*").alias("total_items"),
        F.countDistinct("item_id").alias("distinct_items")
    )
    .orderBy("execution_id", "product_slug")
)
